# Problem Solving 105: Reshape the Managers' Reports

**Goal:** head office wrote the Q2 2026 (April-June) revenue targets in a spreadsheet laid out for people, not for pandas. Reshape it, compare it with what each branch really sold, and write a short note to head office.

**You will practise:** `melt` (wide to long), `merge` on two keys, `pivot_table` (long to wide), and judging what a blank cell means.

**What to submit:** the link to this notebook in your GitHub repository, and your note to head office pasted in the text box.

In [ ]:
# Setup: run this cell
import pandas as pd
import numpy as np

sales = pd.read_csv("https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch2_sales.csv")
sales["Date"] = pd.to_datetime(sales["Date"], format="%Y-%m-%d")

# Head office's target sheet, exactly as typed in the spreadsheet (FCFA)
targets_wide = pd.DataFrame({
    "Branch":   ["Nkwen", "Mile 4", "Commercial Avenue", "Bafoussam Centre", "TOTAL"],
    "Apr-26":   [2400000, 2600000, 2600000, np.nan, 7600000],
    "May-26":   [2500000, 2800000, 2800000, 1000000, 9100000],
    "Jun-26":   [2600000, 3000000, 3000000, 1200000, 9800000],
    "Q2 Total": [7500000, 8400000, 8400000, 2200000, 26500000],
})
targets_wide

## Part 1. Wide to long with `melt`
In a **wide** table, one variable (the month) is spread across several columns. Most pandas tools prefer **long** (tidy) data: one row per branch per month.

Before you melt, look at the sheet again. Two things in it are there for human readers and would **double-count** after melting. Remove them first.

In [ ]:
# Step 1.1 - Remove the parts made for human readers
# TODO: targets = targets_wide[...]  (drop the TOTAL row)
# TODO: targets = targets.drop(columns=[...])  (drop the total column)

In [ ]:
# Step 1.2 - Melt: one row per Branch and Month
# TODO: targets_long = targets.melt(id_vars="Branch", var_name="Month", value_name="Target_FCFA")
# TODO: turn the text "Apr-26" into a real month: pd.to_datetime(targets_long["Month"], format="%b-%y").dt.to_period("M")

In [ ]:
# Self-check 1
assert "TOTAL" not in set(targets_long["Branch"]), "The TOTAL row is still there: it would double the targets."
assert len(targets_long) == 12, f"Expected 4 branches x 3 months = 12 rows, got {len(targets_long)}."
assert str(targets_long["Month"].dtype) == "period[M]", "Month should be a monthly Period, like 2026-04."
print("Part 1 self-check passed.")

## Part 2. Actual sales, in the same shape
Build the actual revenue per branch per month for April-June 2026 from `sales`, then put targets and actuals side by side.

In [ ]:
# Step 2.1 - Actuals in long format
# TODO: sales["Month"] = sales["Date"].dt.to_period("M")
# TODO: keep April-June only
# TODO: actual_long = (q2.groupby(["Branch", "Month"], as_index=False)["TotalSale_FCFA"].sum()
#                        .rename(columns={"TotalSale_FCFA": "Actual_FCFA"}))

In [ ]:
# Step 2.2 - Put them together and compute % of target
# TODO: compare = targets_long.merge(actual_long, on=["Branch", "Month"], how="left")
# TODO: compare["Pct_of_target"] = (compare["Actual_FCFA"] / compare["Target_FCFA"] * 100).round(1)
# Think: what should happen for a month with NO target? Should a blank become 0?

In [ ]:
# Self-check 2
assert len(compare) == 12, "The merge changed the number of rows. Check the keys Branch and Month."
assert compare["Actual_FCFA"].notna().all(), "Some actuals did not match. Are both Month columns monthly Periods?"
assert not np.isinf(compare["Pct_of_target"]).any(), "Infinite % found: a blank target was turned into 0."
total_actual = sales.loc[sales["Date"].between("2026-04-01", "2026-06-30"), "TotalSale_FCFA"].sum()
assert compare["Actual_FCFA"].sum() == total_actual, "Branch-month actuals do not add up to total Q2 sales."
print("Part 2 self-check passed.")

## Part 3. Long to wide with `pivot_table`, for the report
Managers read a grid: branches down the side, months across the top.

In [ ]:
# Step 3.1 - Report table
# TODO: report = compare.pivot_table(index="Branch", columns="Month", values="Pct_of_target")
# TODO: add a Q2 column for each branch: total actual / total target, using only months that HAVE a target
# TODO: sort the branches in a sensible order and show the table

## Part 4. Note to head office
Write 5-8 sentences in plain English (no code words):
1. Which branches met their targets, and in which months.
2. One thing the quarter total hides (look month by month).
3. At least **one question that needs a business decision** before this report can be final (for example: what does a blank target mean?). Say what you assumed for now.

*(write here, then paste the same note in the submission text box)*

## Before you submit
- [ ] Runtime > Restart session and run all: no errors, and both self-checks print "passed".
- [ ] `report` shows branches as rows and months as columns, plus a Q2 column.
- [ ] Your note answers the three points in Part 4 and uses numbers from your table.
- [ ] The notebook is saved in your GitHub repository and the link opens for someone who is not logged in.